In [23]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

In [34]:
# inputs
state = "KS"
renewable_type = "wind"
min_voltage = 115

# distance threshold 
distance_threshold = 3

## Building Base Map

In [25]:
# getting the US census data

url_0 = "https://www2.census.gov/geo/tiger/GENZ2024/shp/"
url_county = "cb_2024_us_county_500k.zip"
url_state = "cb_2024_us_state_500k.zip"

# state and counties
states = gpd.read_file(url_0 + url_state)
counties = gpd.read_file(url_0 + url_county)

# crs
states = states.to_crs(epsg=2163)
counties = counties.to_crs(epsg=2163)

# selecting state
selected_state = states[states["STUSPS"] == state].copy()

# state name
state_name = selected_state["NAME"].iloc[0]
statefp = selected_state["STATEFP"].iloc[0]

# counties
state_counties = counties[counties["STATEFP"] == statefp].copy()
state_counties = state_counties.sort_values("NAME").reset_index(drop=True)

# boundary
state_boundary = selected_state.geometry.iloc[0]

## Renewable Counts

In [29]:
# select renewable dataset
if renewable_type == "wind":
    renewables = gpd.read_file("wind energy/data/wind_turbines.geojson")
    renewables = renewables[["p_name", "geometry"]].copy()

elif renewable_type == "solar":
    renewables = gpd.read_file("wind energy/data/wind_turbines.geojson")
    renewables = renewables[["geometry"]].copy()

elif renewable_type == "hydro":
    renewables = gpd.read_file("wind energy/data/wind_turbines.geojson")
    renewables = renewables[["geometry"]].copy()

# crs
renewables = renewables.to_crs(epsg=2163)

# select locations within the state
renewables = renewables[renewables.geometry.intersects(state_boundary)].copy()

In [30]:
# spatial join
renewables = gpd.sjoin(renewables, state_counties[["NAME", "geometry"]], how="inner", predicate="within")

# rename county column
renewables = renewables.rename(columns={"NAME": "county"})

# filter column and clean columns
renewables = renewables.drop(columns=["index_right"])
renewables = renewables.reset_index(drop=True)

print(renewables["county"].value_counts())

county
Ford          464
Gray          342
Clark         284
Kingman       284
Pratt         227
Harper        203
Republic      202
Sumner        184
Thomas        177
Marion        171
Lincoln       165
Marshall      144
Neosho        139
Ellsworth     124
Nemaha        120
Ellis         115
Elk           111
Butler        100
Coffey         95
Ness           94
Barber         92
Kiowa          77
Haskell        74
Cloud          70
Grant          61
Allen          59
Bourbon        56
Lyon           53
Wichita        33
Washington     32
Rush           21
Trego          17
Crawford       14
Osage           9
Gove            1
Harvey          1
Name: count, dtype: int64


## Transmission Line Proximity

In [ ]:
# transmission files based on minimum voltage
transmission_files = {
    69: [
        "transmission line/data/transmission_69_114.geojson",
        "transmission line/data/transmission_115_229.geojson",
        "transmission line/data/transmission_230_plus.geojson"
    ],

    115: [
        "transmission line/data/transmission_115_229.geojson",
        "transmission line/data/transmission_230_plus.geojson"
    ],

    230: [
        "transmission line/data/transmission_230_plus.geojson"
    ]
}

In [33]:
# load data
transmission_list = []

# Include transmission near the state's borders
search_area = state_boundary.buffer(50000)

for file in transmission_files[min_voltage]:
    lines = gpd.read_file(file)
    lines = lines.to_crs(epsg=2163)
    lines = lines[lines.geometry.intersects(search_area)].copy()
    transmission_list.append(lines[["geometry"]])

# combine selected datasets into one GeoDataFrame
transmission = gpd.GeoDataFrame(
    pd.concat(transmission_list, ignore_index=True),
    geometry="geometry",
    crs="EPSG:2163"
)

In [36]:
# Create a 3 km buffer around transmission lines
transmission_buffer = transmission.copy()
transmission_buffer["geometry"] = transmission_buffer.geometry.buffer(distance_threshold * 1000)

In [38]:
# renewables that intersect transmission buffer
near_renewables = gpd.sjoin(renewables, transmission_buffer[["geometry"]], how="inner", predicate="intersects")

# Classify all renewable locations
renewables["near"] = renewables.index.isin(near_renewables.index)

# Check results
print("Total locations:", len(renewables))
print("Near transmission:", renewables["near"].sum())
print("Far from transmission:", (~renewables["near"]).sum())

Total locations: 4415
Near transmission: 1895
Far from transmission: 2520
